## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Safe mount: skips if already mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print("✓ Drive mounted.")
else:
    print("✓ Drive already mounted — skipping.")

## 2. Imports and environment check

In [ ]:
import json, random, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, Activation, Add,
    GlobalAveragePooling1D, Dense, LSTM, Dropout, Permute, concatenate
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import Callback, ReduceLROnPlateau, ModelCheckpoint
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

print("TensorFlow :", tf.__version__)
print("NumPy      :", np.__version__)
print("GPU        :", tf.config.list_physical_devices('GPU'))

## 3. Configuration  

In [ ]:
DATASET_NAME = "ECG5000"      # name of any UCR dataset with >= 400 training series
SEEDS        = [0, 1, 2, 3]   # seeds for LSTM-FCN Vanilla and CurAT-LSTM-FCN

DRIVE_ROOT = "/content/drive/MyDrive"
TRAIN_PATH = f"{DRIVE_ROOT}/{DATASET_NAME}_TRAIN.txt"
TEST_PATH  = f"{DRIVE_ROOT}/{DATASET_NAME}_TEST.txt"
CKPT_DIR   = f"{DRIVE_ROOT}/UCR_Datasets/{DATASET_NAME}_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

assert os.path.exists(TRAIN_PATH) and os.path.exists(TEST_PATH), \
    f"TRAIN/TEST file not found on Drive for {DATASET_NAME}"
print("Dataset :", DATASET_NAME, "| seeds:", SEEDS)
print("Outputs :", CKPT_DIR)

## 4. Attack functions (FGSM · BIM · PGD · C&W-L2)


In [ ]:
def fgsm_attack(model, X, Y_true, epsilon, batch_size=None):
    if batch_size is None: batch_size = ATK_BATCH
    X_adv = np.zeros_like(X)
    for s in range(0, len(X), batch_size):
        e  = min(s + batch_size, len(X))
        Xb = tf.Variable(tf.cast(X[s:e], tf.float32))
        Yb = tf.cast(Y_true[s:e], tf.float32)
        with tf.GradientTape() as tape:
            loss = tf.keras.losses.categorical_crossentropy(Yb, model(Xb, training=False))
        X_adv[s:e] = (Xb + epsilon * tf.sign(tape.gradient(loss, Xb))).numpy()
    return X_adv


def bim_attack(model, X, Y_true, epsilon, alpha=None, num_iter=100, batch_size=None):
    if batch_size is None: batch_size = ATK_BATCH
    if alpha is None: alpha = epsilon / 10.0
    X_adv = X.copy().astype(np.float32)
    for s in range(0, len(X), batch_size):
        e  = min(s + batch_size, len(X))
        Xb = X[s:e].astype(np.float32)
        Yb = tf.cast(Y_true[s:e], tf.float32)
        Xa = Xb.copy()
        for _ in range(num_iter):
            Xt = tf.Variable(tf.cast(Xa, tf.float32))
            with tf.GradientTape() as tape:
                loss = tf.keras.losses.categorical_crossentropy(Yb, model(Xt, training=False))
            Xa = Xa + alpha * tf.sign(tape.gradient(loss, Xt)).numpy()
            Xa = np.clip(Xa, Xb - epsilon, Xb + epsilon)
        X_adv[s:e] = Xa
    return X_adv


def pgd_attack(model, X, Y_true, epsilon, alpha=None, num_iter=100, batch_size=None):
    # Single run with a uniform random start in the L-inf ball; the final iterate is returned.
    if batch_size is None: batch_size = ATK_BATCH
    if alpha is None: alpha = epsilon / 10.0
    X_adv = X.copy().astype(np.float32)
    for s in range(0, len(X), batch_size):
        e  = min(s + batch_size, len(X))
        Xb = X[s:e].astype(np.float32)
        Yb = tf.cast(Y_true[s:e], tf.float32)
        noise = np.random.uniform(-epsilon, epsilon, Xb.shape).astype(np.float32)
        Xa    = np.clip(Xb + noise, Xb - epsilon, Xb + epsilon)
        for _ in range(num_iter):
            Xt = tf.Variable(tf.cast(Xa, tf.float32))
            with tf.GradientTape() as tape:
                loss = tf.keras.losses.categorical_crossentropy(Yb, model(Xt, training=False))
            Xa = Xa + alpha * tf.sign(tape.gradient(loss, Xt)).numpy()
            Xa = np.clip(Xa, Xb - epsilon, Xb + epsilon)
        X_adv[s:e] = Xa
    return X_adv


def build_logit_model(model):

    penult    = model.layers[-2].output
    logit_out = Dense(model.layers[-1].units, use_bias=True, name='logits_out')(penult)
    lm = Model(inputs=model.input, outputs=logit_out)
    lm.get_layer('logits_out').set_weights(model.layers[-1].get_weights())
    return lm


def cw_l2_attack(model, X, Y_true, c=1.0, kappa=0.0,
                 num_iter=200, lr=0.01, batch_size=64):
    lm      = build_logit_model(model)
    X_adv   = np.zeros_like(X)
    n_batch = int(np.ceil(len(X) / batch_size))
    for b in range(n_batch):
        Xb  = X[b*batch_size:(b+1)*batch_size].astype(np.float32)
        Yb  = Y_true[b*batch_size:(b+1)*batch_size].astype(np.float32)
        n   = len(Xb)
        delta = tf.Variable(tf.zeros_like(Xb), trainable=True)
        opt   = tf.keras.optimizers.Adam(learning_rate=lr)
        for _ in range(num_iter):
            with tf.GradientTape() as tape:
                Xa    = tf.constant(Xb) + delta
                logit = lm(Xa, training=False)
                tl    = tf.reduce_sum(logit * tf.constant(Yb), axis=1)             # true-class logit
                ol    = tf.reduce_max(logit - 1e9 * tf.constant(Yb), axis=1)       # best other logit
                fl    = tf.maximum(tl - ol + kappa, tf.zeros(n))
                l2    = tf.reduce_sum(tf.square(delta), axis=[1, 2])
                loss  = tf.reduce_mean(l2 + c * fl)
            opt.apply_gradients(zip(tape.gradient(loss, [delta]), [delta]))
        X_adv[b*batch_size:(b+1)*batch_size] = (Xb + delta.numpy())
        if (b+1) % 20 == 0:
            print(f"  C&W: batch {b+1}/{n_batch}")
    return X_adv.astype(np.float32)

print("✓ Attack functions defined.")

## 5. Training helpers

In [ ]:
def set_all_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception as e:
        print("enable_op_determinism not available:", e)


def save_history(h, name):
    path = f"{CKPT_DIR}/{name}_history.json"
    with open(path, 'w') as f: json.dump(h, f)
    print(f"  [Drive] History saved: {name}_history.json")


def load_history(name):
    path = f"{CKPT_DIR}/{name}_history.json"
    if os.path.exists(path):
        with open(path) as f: return json.load(f)
    return None


def dummy_hist():
    return {'loss': [0], 'val_loss': [0], 'accuracy': [0], 'val_accuracy': [0]}


def train_standard(model, name, label, seed=None):
    # seed=None: ResNet / FCN / LSTM (trained once, single unseeded run).
    # seed=k   : LSTM-FCN Vanilla (one model per seed).
    suffix    = f"_seed{seed}" if seed is not None else ""
    drive_w   = f"{CKPT_DIR}/{name}{suffix}.weights.h5"
    local_w   = f"/content/best_{name}{suffix}.weights.h5"
    ckpt_w    = f"{CKPT_DIR}/{name}{suffix}_inprogress.weights.h5"
    ckpt_meta = f"{CKPT_DIR}/{name}{suffix}_inprogress.json"

    if os.path.exists(drive_w):
        print(f"✓ {label} (seed={seed}) weights found on Drive — loading.")
        model.load_weights(drive_w)
        h = load_history(f"{name}{suffix}") or dummy_hist()
        print(f"  Best val acc: {max(h['val_accuracy'])*100:.2f}%")
        return h

    initial_epoch, prior_hist = 0, {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
    if os.path.exists(ckpt_w) and os.path.exists(ckpt_meta):
        with open(ckpt_meta) as f: meta = json.load(f)
        initial_epoch = meta['epoch']; prior_hist = meta['history']
        model.load_weights(ckpt_w)
        print(f"✓ Resuming {label} (seed={seed}) from epoch {initial_epoch} (Drive checkpoint found).")

    print("=" * 60)
    print(f"  Training {label} (seed={seed})")
    print(f"  Epochs: {EPOCHS}  |  starting at epoch {initial_epoch}")
    print("=" * 60)

    class DriveCheckpoint(Callback):
        def __init__(self, every=25, hist=None):
            super().__init__(); self.every = every; self.hist = hist
        def on_epoch_end(self, epoch, logs=None):
            for k, v in (logs or {}).items():
                self.hist.setdefault(k, []).append(float(v))
            if (epoch + 1) % self.every == 0:
                self.model.save_weights(ckpt_w)
                with open(ckpt_meta, 'w') as f:
                    json.dump({'epoch': epoch + 1, 'history': self.hist}, f)

    dc_cb = DriveCheckpoint(every=25, hist=prior_hist)
    cbs = [
        ReduceLROnPlateau(monitor='loss', factor=0.5, patience=LR_PATIENCE, min_lr=1e-6, verbose=0),
        ModelCheckpoint(local_w, monitor='val_accuracy', save_best_only=True,
                        save_weights_only=True, verbose=0),
        dc_cb,
    ]
    model.fit(X_train, y_train_cat,
              epochs=EPOCHS, batch_size=BATCH_SIZE, initial_epoch=initial_epoch,
              validation_data=(X_test, y_test_cat),
              callbacks=cbs, verbose=1)

    model.load_weights(local_w)
    model.save_weights(drive_w)
    h = dc_cb.hist
    save_history(h, f"{name}{suffix}")
    for p in [ckpt_w, ckpt_meta]:
        if os.path.exists(p): os.remove(p)
    print(f"\n  ✓ {label} (seed={seed}) done. Best val acc = {max(h['val_accuracy'])*100:.2f}%")
    print("  Weights + history saved to Drive.")
    return h

## 6. Data loading and per-dataset configuration


In [ ]:
def load_dataset(DATASET_NAME, TRAIN_PATH, TEST_PATH):
    def load_ucr_raw(path):
        df = pd.read_csv(path, sep=r'\s+', header=None, engine='python')
        return df.iloc[:, 1:].values.astype(np.float32), df.iloc[:, 0].values

    X_train_raw, y_train_raw = load_ucr_raw(TRAIN_PATH)
    X_test_raw,  y_test_raw  = load_ucr_raw(TEST_PATH)

    n_train = X_train_raw.shape[0]
    assert n_train >= 400, f"Only {n_train} training samples — minimum is 400."

    le = LabelEncoder()
    le.fit(np.concatenate([y_train_raw, y_test_raw]))
    y_train_int = le.transform(y_train_raw)
    y_test_int  = le.transform(y_test_raw)
    num_classes   = len(le.classes_)
    series_length = X_train_raw.shape[1]

    X_train = X_train_raw[:, :, np.newaxis]
    X_test  = X_test_raw[:,  :, np.newaxis]
    X_train = (X_train - X_train.mean(axis=1, keepdims=True)) / (X_train.std(axis=1, keepdims=True) + 1e-8)
    X_test  = (X_test  - X_test.mean(axis=1,  keepdims=True)) / (X_test.std(axis=1,  keepdims=True) + 1e-8)

    y_train_cat = to_categorical(y_train_int, num_classes)
    y_test_cat  = to_categorical(y_test_int,  num_classes)

    input_shape = (series_length, 1)
    EPSILON     = float(np.clip(0.1 * np.sqrt(24 / series_length), 0.01, 0.10))
    ATK_BATCH   = 64 if series_length > 1000 else (128 if series_length > 500 else 256)

    if n_train < 1000:
        EPOCHS, LR_PATIENCE = 1500, 100
    elif n_train < 3000:
        EPOCHS, LR_PATIENCE = 1000, 75
    else:
        EPOCHS, LR_PATIENCE = 500, 50
    BATCH_SIZE = 64

    print(f"Dataset       : {DATASET_NAME}")
    print(f"Train shape   : {X_train.shape}  |  Test shape: {X_test.shape}")
    print(f"Classes       : {num_classes}")
    print(f"Series length : {series_length}")
    print(f"Epochs        : {EPOCHS}  |  Batch: {BATCH_SIZE}  |  ε: {EPSILON:.4f}")

    return {
        'X_train': X_train, 'y_train_cat': y_train_cat, 'y_train_int': y_train_int,
        'X_test': X_test, 'y_test_cat': y_test_cat, 'y_test_int': y_test_int,
        'num_classes': num_classes, 'series_length': series_length,
        'input_shape': input_shape, 'EPSILON': EPSILON, 'ATK_BATCH': ATK_BATCH,
        'EPOCHS': EPOCHS, 'LR_PATIENCE': LR_PATIENCE, 'BATCH_SIZE': BATCH_SIZE,
    }

## 7. Baseline architectures: ResNet, FCN, LSTM

In [ ]:
def build_fixed_models(input_shape, num_classes):
    def res_block(x, f):
        c = Conv1D(f, 8, padding='same', use_bias=False)(x)
        c = BatchNormalization()(c); c = Activation('relu')(c)
        c = Conv1D(f, 5, padding='same', use_bias=False)(c)
        c = BatchNormalization()(c); c = Activation('relu')(c)
        c = Conv1D(f, 3, padding='same', use_bias=False)(c)
        c = BatchNormalization()(c)
        s = Conv1D(f, 1, padding='same', use_bias=False)(x)
        s = BatchNormalization()(s)
        return Activation('relu')(Add()([s, c]))

    def build_resnet():
        inp = Input(shape=input_shape)
        x   = res_block(inp, 64); x = res_block(x, 128); x = res_block(x, 128)
        x   = GlobalAveragePooling1D()(x)
        return Model(inp, Dense(num_classes, activation='softmax')(x), name='ResNet')

    def build_fcn():
        inp = Input(shape=input_shape)
        x   = Conv1D(128, 8, padding='same', use_bias=False)(inp)
        x   = BatchNormalization()(x); x = Activation('relu')(x)
        x   = Conv1D(256, 5, padding='same', use_bias=False)(x)
        x   = BatchNormalization()(x); x = Activation('relu')(x)
        x   = Conv1D(128, 3, padding='same', use_bias=False)(x)
        x   = BatchNormalization()(x); x = Activation('relu')(x)
        x   = GlobalAveragePooling1D()(x)
        return Model(inp, Dense(num_classes, activation='softmax')(x), name='FCN')

    def build_lstm():
        inp = Input(shape=input_shape)
        x   = LSTM(128, return_sequences=True)(inp)
        x   = Dropout(0.5)(x)
        x   = LSTM(64, return_sequences=False)(x)
        x   = Dropout(0.3)(x)
        return Model(inp, Dense(num_classes, activation='softmax')(x), name='LSTM')

    resnet_model = build_resnet()
    fcn_model    = build_fcn()
    lstm_model   = build_lstm()

    for m in [resnet_model, fcn_model, lstm_model]:
        m.compile(Adam(1e-3), 'categorical_crossentropy', metrics=['accuracy'])

    print("✓ ResNet, FCN, LSTM built:")
    for m in [resnet_model, fcn_model, lstm_model]:
        print(f"  {m.name:<25} params={m.count_params():,}")

    return resnet_model, fcn_model, lstm_model

## 8. LSTM-FCN models

In [ ]:
def build_seed_models(input_shape, num_classes, seed):
    set_all_seeds(seed)

    def build_lstm_fcn(name):
        inp      = Input(shape=input_shape)
        x        = Conv1D(128, 8, padding='same', use_bias=False)(inp)
        x        = BatchNormalization()(x); x = Activation('relu')(x)
        x        = Conv1D(256, 5, padding='same', use_bias=False)(x)
        x        = BatchNormalization()(x); x = Activation('relu')(x)
        x        = Conv1D(128, 3, padding='same', use_bias=False)(x)
        x        = BatchNormalization()(x); x = Activation('relu')(x)
        fcn_out  = GlobalAveragePooling1D()(x)
        lstm_in  = Permute((2, 1))(inp)                  # dimension shuffle
        lstm_out = LSTM(8)(lstm_in)
        lstm_out = Dropout(0.8)(lstm_out)
        merged   = concatenate([fcn_out, lstm_out])
        return Model(inp, Dense(num_classes, activation='softmax')(merged), name=name)

    lstmfcn_vanilla = build_lstm_fcn('LSTM_FCN_Vanilla')
    lstmfcn_vanilla.compile(Adam(1e-3), 'categorical_crossentropy', metrics=['accuracy'])

    lstmfcn_robust = build_lstm_fcn('LSTM_FCN_Robust')
    lstmfcn_robust.compile(Adam(1e-3), 'categorical_crossentropy', metrics=['accuracy'])

    return lstmfcn_vanilla, lstmfcn_robust

## 9. Generate (or load cached) adversarial test sets


In [ ]:
def generate_attacks_for_model(model, model_key, X_test, y_test_cat,
                               EPSILON, series_length, CKPT_DIR, seed_tag=""):
    ATTACK_GEN_SEED = 999
    adv_dir = f"{CKPT_DIR}/adv_arrays/{model_key}{seed_tag}"
    os.makedirs(adv_dir, exist_ok=True)

    paths = {
        'fgsm': f"{adv_dir}/X_fgsm.npy",
        'bim':  f"{adv_dir}/X_bim.npy",
        'pgd':  f"{adv_dir}/X_pgd.npy",
        'cw':   f"{adv_dir}/X_cw.npy",
    }

    if all(os.path.exists(p) for p in paths.values()):
        print(f"✓ [{model_key}{seed_tag}] adversarial arrays already exist — loading, NOT regenerating.")
        X_fgsm = np.load(paths['fgsm']); X_bim = np.load(paths['bim'])
        X_pgd  = np.load(paths['pgd']);  X_cw  = np.load(paths['cw'])
        linf = float(np.max(np.abs(X_fgsm - X_test)))
        assert linf <= EPSILON + 1e-4, (
            f"Cached arrays have L-inf = {linf:.4f} > current ε = {EPSILON:.4f}. "
            f"They were generated with a different ε — delete {CKPT_DIR}/adv_arrays and re-run.")
    else:
        set_all_seeds(ATTACK_GEN_SEED)
        print(f"Generating WHITE-BOX attacks against [{model_key}{seed_tag}] (ε={EPSILON:.4f})...")

        print("  FGSM...", end=" ")
        X_fgsm = fgsm_attack(model, X_test, y_test_cat, epsilon=EPSILON)
        print("done")

        print("  BIM (100 iter)...", end=" ")
        X_bim = bim_attack(model, X_test, y_test_cat, epsilon=EPSILON, num_iter=100)
        print("done")

        print("  PGD (100 iter)...", end=" ")
        X_pgd = pgd_attack(model, X_test, y_test_cat, epsilon=EPSILON, num_iter=100)
        print("done")

        cw_bs = 32 if series_length > 1000 else 64
        print(f"  C&W (200 iter, batch={cw_bs})...")
        X_cw = cw_l2_attack(model, X_test, y_test_cat,
                            c=1.0, kappa=0.0, num_iter=200, lr=0.01, batch_size=cw_bs)
        print("  C&W done")

        np.save(paths['fgsm'], X_fgsm); np.save(paths['bim'], X_bim)
        np.save(paths['pgd'], X_pgd);   np.save(paths['cw'], X_cw)
        print(f"✓ [{model_key}{seed_tag}] all 4 adversarial arrays saved: {adv_dir}")

    print("  Perturbation norms:")
    for name, Xa in [('FGSM', X_fgsm), ('BIM', X_bim), ('PGD', X_pgd), ('C&W', X_cw)]:
        d = Xa - X_test
        print(f"    {name:<6}: L∞={np.max(np.abs(d)):.4f}  "
              f"L2(avg)={np.mean(np.sqrt(np.sum(d**2, axis=(1, 2)))):.4f}")

    return X_fgsm, X_bim, X_pgd, X_cw

## 10. CurAT training (three-phase curriculum)



In [ ]:
def train_curat(seed, DATASET_NAME, CKPT_DIR, lstmfcn_vanilla, lstmfcn_robust,
                X_train, y_train_cat, X_test, y_test_cat, y_test_int,
                series_length, num_classes, EPOCHS, LR_PATIENCE, BATCH_SIZE, ATK_BATCH):

    ADV_EPS = float(np.clip(0.1 * np.sqrt(24 / series_length), 0.01, 0.10))
    BIM_TRAIN_ITER = int(np.clip(np.round(5 + 5 * (series_length - 200) / 1600), 5, 10))
    BIM_ALPHA      = ADV_EPS / BIM_TRAIN_ITER
    difficulty = float(np.log1p(num_classes) * np.sqrt(series_length) / np.sqrt(len(X_train)))
    MAX_ADV    = float(np.clip(0.70 - 0.15 * np.log1p(difficulty), 0.35, 0.70))
    WARMUP = min(750, int(EPOCHS * 0.50))
    RAMP   = min(900, int(EPOCHS * 0.75))

    drive_robust_w = f"{CKPT_DIR}/robust_fixed_seed{seed}.weights.h5"
    van_w          = f"{CKPT_DIR}/vanilla_seed{seed}.weights.h5"
    best_robust_w  = f"{CKPT_DIR}/robust_fixed_seed{seed}_best_inprogress.weights.h5"
    ckpt_robust_w  = f"{CKPT_DIR}/robust_fixed_seed{seed}_inprogress.weights.h5"
    ckpt_meta      = f"{CKPT_DIR}/robust_fixed_seed{seed}_inprogress.json"

    print(f"Dataset : {DATASET_NAME}  |  T={series_length}  |  C={num_classes}  |  "
          f"n_train={len(X_train)}  |  seed={seed}")
    print(f"ε={ADV_EPS:.4f}  |  BIM_iter={BIM_TRAIN_ITER}  |  MAX_ADV={MAX_ADV:.2f}  |  difficulty={difficulty:.3f}")
    print(f"Phase1(clean): 0→{WARMUP}  |  Phase2(ramp): {WARMUP}→{RAMP}  |  Phase3(FGSM+BIM): {RAMP}→{EPOCHS}")

    def adv_val_acc_fast(model, eps):
        preds = np.zeros(len(X_test), dtype=np.int32)
        for s in range(0, len(X_test), ATK_BATCH):
            e  = min(s + ATK_BATCH, len(X_test))
            Xv = tf.Variable(tf.cast(X_test[s:e], tf.float32))
            Yv = tf.cast(y_test_cat[s:e], tf.float32)
            with tf.GradientTape() as tape:
                loss = tf.keras.losses.categorical_crossentropy(Yv, model(Xv, training=False))
            Xf = (Xv + eps * tf.sign(tape.gradient(loss, Xv))).numpy()
            preds[s:e] = np.argmax(model.predict(Xf, verbose=0), axis=1)
        return float(np.mean(preds == y_test_int))

    def real_clean_acc(model):
        return float(np.mean(np.argmax(model.predict(X_test, verbose=0), axis=1) == y_test_int))

    if os.path.exists(drive_robust_w):
        print(f"✓ CurAT weights already exist for seed={seed} — loading, NOT retraining.")
        lstmfcn_robust.load_weights(drive_robust_w)
        rc, ra = real_clean_acc(lstmfcn_robust), adv_val_acc_fast(lstmfcn_robust, ADV_EPS)
        print(f"  Real clean={rc*100:.2f}%  Real FGSM(ε)={ra*100:.2f}%")
        hist_robust = load_history(f'{DATASET_NAME}_robust_fixed_seed{seed}') or {
            'loss': [], 'val_accuracy': [], 'val_acc': [],
            'saved_checkpoint_clean_acc': rc, 'saved_checkpoint_adv_acc': ra
        }
        return lstmfcn_robust, hist_robust

    class CollapseGuard(Callback):
        def __init__(self, n_cls, warmup, patience=150):
            super().__init__()
            self.threshold = (1.0 / n_cls) * 1.5; self.warmup = warmup
            self.patience = patience; self.wait = 0
        def on_epoch_end(self, epoch, logs=None):
            va = logs.get('val_accuracy', 1.0)
            if epoch >= self.warmup and va < self.threshold:
                self.wait += 1
                if self.wait >= self.patience:
                    print(f"\n⚠ CollapseGuard: stopped epoch {epoch+1}")
                    self.model.stop_training = True
            else:
                self.wait = 0

    if os.path.exists(van_w):
        lstmfcn_vanilla.load_weights(van_w)
        lstmfcn_robust.set_weights(lstmfcn_vanilla.get_weights())
        print(f"✓ Warm-start: Vanilla (seed={seed}) → CurAT (seed={seed})")
    else:
        print(f"⚠ Vanilla weights for seed={seed} not found — train Vanilla first")

    set_all_seeds(seed)

    start_epoch = 0
    best_clean = 0.0; best_adv = 0.0
    best_clean_epoch = -1; best_adv_epoch = -1; best_adv_clean_at_save = 0.0
    lr_wait = 0; prev_loss = np.inf; current_lr = 1e-3
    use_bim = False
    adv_log = {'loss': [], 'val_accuracy': [], 'val_acc': []}

    if os.path.exists(ckpt_robust_w) and os.path.exists(ckpt_meta) and os.path.exists(best_robust_w):
        with open(ckpt_meta) as f: meta = json.load(f)
        start_epoch            = meta['epoch']
        best_clean, best_adv   = meta['best_clean'], meta['best_adv']
        best_clean_epoch       = meta['best_clean_epoch']
        best_adv_epoch         = meta['best_adv_epoch']
        best_adv_clean_at_save = meta['best_adv_clean_at_save']
        lr_wait, prev_loss     = meta['lr_wait'], meta['prev_loss']
        current_lr, use_bim    = meta['current_lr'], meta['use_bim']
        adv_log                = meta['adv_log']
        lstmfcn_robust.load_weights(ckpt_robust_w)
        print(f"✓ Resuming CurAT (seed={seed}) from epoch {start_epoch} (Drive checkpoint found).")

    optimizer  = tf.keras.optimizers.Adam(1e-3)
    optimizer.learning_rate.assign(current_lr)
    loss_fn    = tf.keras.losses.CategoricalCrossentropy()
    n_samples  = len(X_train)
    guard = CollapseGuard(num_classes, WARMUP)
    guard.set_model(lstmfcn_robust)

    print("\nTraining started...\n")

    for epoch in range(start_epoch, EPOCHS):
        if epoch < WARMUP:
            adv_ratio, phase_name = 0.0, "Phase1-Clean"
        elif epoch < RAMP:
            adv_ratio = ((epoch - WARMUP) / (RAMP - WARMUP)) * MAX_ADV
            phase_name = "Phase2-Ramp"
        else:
            adv_ratio, phase_name = MAX_ADV, "Phase3-MultiAtk"

        perm = np.random.permutation(n_samples)
        Xs, Ys = X_train[perm], y_train_cat[perm]
        epoch_losses = []

        for s in range(0, n_samples, BATCH_SIZE):
            e = min(s + BATCH_SIZE, n_samples)
            Xb = Xs[s:e].astype(np.float32); Yb = Ys[s:e].astype(np.float32)

            if adv_ratio > 0:
                n_adv = max(1, int(len(Xb) * adv_ratio))
                Xb_orig = Xb[:n_adv]; Yb_adv = Yb[:n_adv]

                if phase_name == "Phase3-MultiAtk" and use_bim:
                    Xa = Xb_orig.copy()
                    for _ in range(BIM_TRAIN_ITER):
                        Xv = tf.Variable(tf.cast(Xa, tf.float32))
                        with tf.GradientTape() as t2:
                            la = tf.keras.losses.categorical_crossentropy(
                                tf.cast(Yb_adv, tf.float32), lstmfcn_robust(Xv, training=False))
                        Xa = Xa + BIM_ALPHA * tf.sign(t2.gradient(la, Xv)).numpy()
                        Xa = np.clip(Xa, Xb_orig - ADV_EPS, Xb_orig + ADV_EPS)
                    Xadv = Xa
                else:
                    Xv = tf.Variable(tf.cast(Xb_orig, tf.float32))
                    with tf.GradientTape() as t2:
                        la = tf.keras.losses.categorical_crossentropy(
                            tf.cast(Yb_adv, tf.float32), lstmfcn_robust(Xv, training=False))
                    Xadv = (Xv + ADV_EPS * tf.sign(t2.gradient(la, Xv))).numpy()

                Xb = np.concatenate([Xadv, Xb[n_adv:]], axis=0)

            with tf.GradientTape() as tape:
                preds = lstmfcn_robust(tf.cast(Xb, tf.float32), training=True)
                loss = loss_fn(tf.cast(Yb, tf.float32), preds)
            grads = tape.gradient(loss, lstmfcn_robust.trainable_variables)
            grads, _ = tf.clip_by_global_norm(grads, clip_norm=1.0)
            optimizer.apply_gradients(zip(grads, lstmfcn_robust.trainable_variables))
            epoch_losses.append(float(loss))

        if phase_name == "Phase3-MultiAtk":
            use_bim = not use_bim          # alternate FGSM / BIM epochs

        ml = float(np.mean(epoch_losses))
        va = float(np.mean(np.argmax(lstmfcn_robust.predict(X_test, verbose=0), axis=1) == y_test_int))
        adv_log['loss'].append(ml); adv_log['val_accuracy'].append(va); adv_log['val_acc'].append(va)

        if epoch < WARMUP:
            if va > best_clean:
                best_clean, best_clean_epoch = va, epoch
                lstmfcn_robust.save_weights(best_robust_w)
        else:
            adv_va = adv_val_acc_fast(lstmfcn_robust, ADV_EPS)
            if adv_va > best_adv:
                best_adv, best_adv_epoch, best_adv_clean_at_save = adv_va, epoch, va
                lstmfcn_robust.save_weights(best_robust_w)

        if ml < prev_loss - 1e-5:
            lr_wait = 0; prev_loss = ml
        else:
            lr_wait += 1
            if lr_wait >= LR_PATIENCE and current_lr > 1e-6:
                current_lr = max(current_lr * 0.5, 1e-6)
                optimizer.learning_rate.assign(current_lr); lr_wait = 0; prev_loss = ml

        if (epoch + 1) % 100 == 0 or epoch == 0:
            atk = "BIM-" + str(BIM_TRAIN_ITER) if (use_bim and phase_name == "Phase3-MultiAtk") else "FGSM"
            print(f"  Ep {epoch+1:>5}/{EPOCHS}  loss={ml:.4f}  val_acc={va*100:.2f}%  "
                  f"mix={adv_ratio:.0%}  atk={atk}  lr={current_lr:.1e}  [{phase_name}]")


        if (epoch + 1) % 25 == 0:
            lstmfcn_robust.save_weights(ckpt_robust_w)
            with open(ckpt_meta, 'w') as f:
                json.dump({'epoch': epoch + 1, 'best_clean': best_clean, 'best_adv': best_adv,
                           'best_clean_epoch': best_clean_epoch, 'best_adv_epoch': best_adv_epoch,
                           'best_adv_clean_at_save': best_adv_clean_at_save,
                           'lr_wait': lr_wait, 'prev_loss': prev_loss, 'current_lr': current_lr,
                           'use_bim': use_bim, 'adv_log': adv_log}, f)

        guard.on_epoch_end(epoch, {'val_accuracy': va})
        if lstmfcn_robust.stop_training:
            break

    lstmfcn_robust.load_weights(best_robust_w)
    lstmfcn_robust.save_weights(drive_robust_w)
    saved_clean_acc = best_adv_clean_at_save if best_adv_epoch > best_clean_epoch else best_clean
    saved_epoch = best_adv_epoch if best_adv_epoch > best_clean_epoch else best_clean_epoch
    adv_log['saved_checkpoint_clean_acc'] = saved_clean_acc
    adv_log['saved_checkpoint_adv_acc']   = best_adv
    adv_log['saved_checkpoint_epoch']     = saved_epoch
    save_history(adv_log, f'{DATASET_NAME}_robust_fixed_seed{seed}')

    for p in [ckpt_robust_w, ckpt_meta, best_robust_w]:
        if os.path.exists(p): os.remove(p)

    print(f"\n✓ Done (seed={seed}).  Saved checkpoint epoch: {saved_epoch}")
    print(f"  Real clean={saved_clean_acc*100:.2f}%  Real adv(FGSM,ε)={best_adv*100:.2f}%")
    print(f"  Saved → {drive_robust_w}")

    return lstmfcn_robust, adv_log

## 11. Evaluate all models and log results


In [ ]:
def evaluate_dataset_seed_whitebox(DATASET_NAME, seed, CKPT_DIR,
                                   resnet_model, fcn_model, lstm_model,
                                   lstmfcn_vanilla, lstmfcn_robust,
                                   X_test, y_test_cat, y_test_int,
                                   EPSILON, series_length,
                                   multiseed_records):

    all_models = {
        'ResNet'           : resnet_model,
        'FCN'              : fcn_model,
        'LSTM'             : lstm_model,
        'LSTM-FCN Vanilla' : lstmfcn_vanilla,
        'CurAT-LSTM-FCN'   : lstmfcn_robust,
    }

    FIXED_MODELS    = ['ResNet', 'FCN', 'LSTM']                        # trained once, attacked once
    PER_SEED_MODELS = ['LSTM-FCN Vanilla', 'CurAT-LSTM-FCN']           # retrained for each seed

    already_logged_fixed = any(
        r['dataset'] == DATASET_NAME and r['model'] in FIXED_MODELS
        for r in multiseed_records
    )
    models_to_eval = {
        mname: model for mname, model in all_models.items()
        if mname not in FIXED_MODELS or not already_logged_fixed
    }

    key_map = {'ResNet': 'resnet', 'FCN': 'fcn', 'LSTM': 'lstm',
               'LSTM-FCN Vanilla': 'vanilla', 'CurAT-LSTM-FCN': 'robust'}

    print(f"Dataset={DATASET_NAME}  seed={seed}  |  Evaluating {len(models_to_eval)} model(s), white-box...")

    for mname, model in models_to_eval.items():
        model_key = key_map[mname]
        seed_tag  = f"_seed{seed}" if mname in PER_SEED_MODELS else ""

        X_fgsm, X_bim, X_pgd, X_cw = generate_attacks_for_model(
            model, model_key, X_test, y_test_cat, EPSILON, series_length, CKPT_DIR, seed_tag)

        preds = {
            'clean': np.argmax(model.predict(X_test, verbose=0), axis=1),
            'fgsm' : np.argmax(model.predict(X_fgsm, verbose=0), axis=1),
            'bim'  : np.argmax(model.predict(X_bim,  verbose=0), axis=1),
            'pgd'  : np.argmax(model.predict(X_pgd,  verbose=0), axis=1),
            'cw'   : np.argmax(model.predict(X_cw,   verbose=0), axis=1),
        }
        ac, fgsm, bim, pgd, cw = [accuracy_score(y_test_int, preds[k])
                                  for k in ['clean', 'fgsm', 'bim', 'pgd', 'cw']]

        logged_seed = None if mname in FIXED_MODELS else seed
        multiseed_records.append({
            'dataset': DATASET_NAME, 'model': mname, 'seed': logged_seed,
            'clean': ac, 'fgsm': fgsm, 'bim': bim, 'pgd': pgd, 'cw': cw,
            'mean_attack': np.mean([fgsm, bim, pgd, cw]),
            'attack_mode': 'White-box',
        })
        print(f"  ✓ {mname:<25} Clean={ac*100:.2f}%")

    snapshot_path = f"{CKPT_DIR}/{DATASET_NAME}_multiseed_records.xlsx"
    pd.DataFrame(multiseed_records).to_excel(snapshot_path, index=False)
    print(f"✓ Snapshot saved: {snapshot_path}")

    return multiseed_records

## 12. Load the dataset and train the baselines (ResNet, FCN, LSTM)

In [ ]:
data = load_dataset(DATASET_NAME, TRAIN_PATH, TEST_PATH)
globals().update(data)     # X_train, y_train_cat, X_test, ..., EPSILON, EPOCHS, ATK_BATCH, ...

resnet_model, fcn_model, lstm_model = build_fixed_models(input_shape, num_classes)

hist_resnet = train_standard(resnet_model, 'resnet', 'ResNet')
hist_fcn    = train_standard(fcn_model,    'fcn',    'FCN')
hist_lstm   = train_standard(lstm_model,   'lstm',   'LSTM')

snapshot_path = f"{CKPT_DIR}/{DATASET_NAME}_multiseed_records.xlsx"
if os.path.exists(snapshot_path):
    multiseed_records = pd.read_excel(snapshot_path).to_dict('records')
    print(f"✓ Resumed {len(multiseed_records)} prior records for {DATASET_NAME}")
else:
    multiseed_records = []

print(f"\n✓ Setup complete for {DATASET_NAME}")

## 13. One full seed: Vanilla → CurAT (warm start) → evaluation

In [ ]:
def run_one_seed(seed):
    print(f"\n--- {DATASET_NAME} | seed={seed} ---")

    lstmfcn_vanilla, lstmfcn_robust = build_seed_models(input_shape, num_classes, seed)

    hist_vanilla = train_standard(lstmfcn_vanilla, 'vanilla', 'LSTM-FCN Vanilla', seed=seed)

    lstmfcn_robust, hist_robust = train_curat(
        seed, DATASET_NAME, CKPT_DIR, lstmfcn_vanilla, lstmfcn_robust,
        X_train, y_train_cat, X_test, y_test_cat, y_test_int,
        series_length, num_classes, EPOCHS, LR_PATIENCE, BATCH_SIZE, ATK_BATCH)

    global multiseed_records
    multiseed_records = evaluate_dataset_seed_whitebox(
        DATASET_NAME, seed, CKPT_DIR, resnet_model, fcn_model, lstm_model,
        lstmfcn_vanilla, lstmfcn_robust, X_test, y_test_cat, y_test_int,
        EPSILON, series_length, multiseed_records)


    last = [r for r in multiseed_records
            if r['dataset'] == DATASET_NAME and r['model'] == 'CurAT-LSTM-FCN' and r['seed'] == seed]
    if last:
        row = last[-1]
        for k in ['clean', 'fgsm', 'bim', 'pgd', 'cw']:
            v = row[k]
            if v == 0.0 or v == 1.0 or (isinstance(v, float) and np.isnan(v)):
                print(f"  ⚠⚠⚠ SANITY FLAG: {DATASET_NAME} seed={seed} CurAT {k}={v} — verify manually!")

    print(f"✓✓✓ {DATASET_NAME} seed={seed} COMPLETE ✓✓✓")

## 14. Run all seeds

In [ ]:
for s in SEEDS:
    run_one_seed(s)

## 15. Inspect the results table

In [ ]:
df = pd.read_excel(f"{CKPT_DIR}/{DATASET_NAME}_multiseed_records.xlsx")

print(f"Total rows: {len(df)}   (expected {3 + 2*len(SEEDS)}: 3 baselines + 2 models x {len(SEEDS)} seeds)")
print("\nSeeds present per model:")
print(df.groupby('model')['seed'].apply(lambda x: sorted(x.dropna().unique().tolist())))
print("\nFull table:")
print(df.to_string(index=False))